# `extract_openmeteo.ipynb` - Extraccion resiliente de la API de Open-Meteo

Obtiene, para cada ciudad del catalogo, el pronostico horario a 14 dias
(zona horaria UTC) de las cinco variables requeridas por el enunciado:
`temperature_2m`, `relative_humidity_2m`, `precipitation`, `wind_speed_10m`
y `surface_pressure`, desde `https://api.open-meteo.com/v1/forecast`.

Cubre el requisito funcional 1 del enunciado ("Ingesta Resiliente"): el
modulo debe capturar fallos transitorios de red o limites de peticiones
(codigos HTTP 429, 500, 502, 503, 504) mediante reintentos automaticos con
retroceso progresivo antes de descartar una llamada.

> **Decision de diseno (adaptador urllib3)**: la politica de reintentos se
> implementa montando un `HTTPAdapter` de `requests` con una estrategia
> `Retry` de `urllib3` (`status_forcelist` + `backoff_factor`), tal y como
> pide explicitamente el enunciado ("configuracion de adaptadores HTTP,
> control de timeouts y politicas de reintento"). Esto delega el
> reintento con retroceso progresivo en la propia libreria urllib3, en
> lugar de un bucle manual.

> **Decision de diseno (dataset de FALLBACK)**: a diferencia de un dato de
> referencia estatico (poblacion, renta, etc.), un pronostico
> meteorologico no tiene un "valor real" fijo que se pueda guardar de
> antemano como respaldo. Siguiendo el principio de "datos reales, nunca
> inventados", el FALLBACK de este pipeline es **el ultimo lote Bronze
> real** obtenido con exito para esa misma ciudad en una ejecucion
> anterior (nunca un valor sintetico generado en el momento). Si no existe
> ningun lote Bronze previo para una ciudad (primera ejecucion sin red),
> esa ciudad se omite en la corrida actual y queda registrado como error
> en el log, en lugar de inventar un valor.

> **Nota sobre el entorno de desarrollo**: `config.ipynb` expone
> `API_URL_BASE`, que respeta la variable de entorno
> `CLIMATE_ENGINE_API_URL_OVERRIDE`. En el entorno de desarrollo de este
> proyecto no hay salida a internet, por lo que las pruebas rapidas de
> este notebook se han validado tambien contra `tests/mock_openmeteo_server.py`
> (mismo esquema JSON que la API real), documentado en el README. En un
> entorno con conexion a internet, el pipeline funciona exactamente igual
> sin definir esa variable.

> Depende de: `config.ipynb`, `logging_config.ipynb`.

In [1]:
import os
if os.getcwd().endswith("climate_engine"):
    os.chdir("..")
%run climate_engine/config.ipynb
%run climate_engine/logging_config.ipynb

Fichero de configuracion: /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/config.yaml
Raiz del proyecto:        /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco
Ubicaciones cargadas (5): ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
Variables horarias:       ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'surface_pressure']
Ventana de prediccion:    14 dias, zona horaria UTC
Politica de reintentos:   3 reintentos, backoff x2, codigos [429, 500, 502, 503, 504]
Rango temperatura valido: (-50.0, 60.0)
Rango humedad valido:     (0.0, 100.0)
Ruta bronze:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/bronze  (existe: True)
Ruta silver:  /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/silver  (existe: True)
Ruta gold:    /root/project/Entregable_Proyecto_ClimateEngine_Ortega_Calvo_Francisco/data/gold  (existe: True)
URL base API: http://127.0.0.1:87

In [2]:
import json
import glob
from datetime import datetime, timezone

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Funcion principal

In [3]:
def crear_sesion_resiliente():
    """Crea una sesion de requests con un adaptador HTTP de urllib3
    configurado con reintentos automaticos y retroceso progresivo.

    Reintenta automaticamente ante codigos HTTP transitorios
    (429, 500, 502, 503, 504) y ante errores de conexion, con backoff
    exponencial (API_BACKOFF_FACTOR), antes de darse por vencido.

    Devuelve: objeto requests.Session listo para usar.
    """
    sesion = requests.Session()
    estrategia_reintento = Retry(
        total=API_MAX_REINTENTOS,
        connect=API_MAX_REINTENTOS,
        read=API_MAX_REINTENTOS,
        status=API_MAX_REINTENTOS,
        backoff_factor=API_BACKOFF_FACTOR,
        status_forcelist=API_STATUS_FORCELIST,
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adaptador = HTTPAdapter(max_retries=estrategia_reintento)
    sesion.mount("https://", adaptador)
    sesion.mount("http://", adaptador)
    return sesion


def construir_parametros(lat, lon):
    """Construye el diccionario de parametros de query para el endpoint
    /v1/forecast de Open-Meteo, con las variables y la ventana temporal
    exigidas por el enunciado.
    """
    return {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(API_VARIABLES_HORARIAS),
        "forecast_days": API_FORECAST_DAYS,
        "timezone": API_TIMEZONE,
    }


def ruta_ultimo_bronze(nombre_ciudad):
    """Localiza el fichero Bronze mas reciente ya guardado para una ciudad
    (usado como fuente del FALLBACK).

    Decision de diseno: se ordena por el timestamp_ingesta_utc REAL
    guardado dentro de cada fichero (metadata), no por orden alfabetico
    del nombre de fichero. El id_lote es un hash aleatorio
    (uuid.uuid4().hex[:12]), asi que ordenar los nombres de fichero
    alfabeticamente no garantiza el mas reciente: solo el timestamp de
    ingesta refleja el orden cronologico real.

    Devuelve None si no existe ninguno.
    """
    patron = str(RUTA_BRONZE / f"ciudad={nombre_ciudad}" / "**" / "*.json")
    candidatos = glob.glob(patron, recursive=True)
    if not candidatos:
        return None

    def _timestamp_ingesta(ruta_fichero):
        with open(ruta_fichero, "r", encoding="utf-8") as f:
            return datetime.fromisoformat(json.load(f)["metadata"]["timestamp_ingesta_utc"])

    return max(candidatos, key=_timestamp_ingesta)


def cargar_fallback_desde_bronze(nombre_ciudad):
    """Recupera el ultimo cuerpo JSON real (no inventado) guardado en
    Bronze para esta ciudad, para usarlo como FALLBACK cuando la API no
    responde. Devuelve None si no hay ningun lote previo disponible.
    """
    ruta = ruta_ultimo_bronze(nombre_ciudad)
    if ruta is None:
        return None
    with open(ruta, "r", encoding="utf-8") as f:
        registro = json.load(f)
    logger.warning(
        f"Usando como FALLBACK el ultimo lote Bronze real disponible para "
        f"{nombre_ciudad}: {ruta}"
    )
    return registro["datos_api"]


def obtener_datos_ciudad(sesion, nombre_ciudad, lat, lon):
    """Obtiene el pronostico horario a 14 dias de Open-Meteo para una
    ciudad concreta.

    Aplica el patron de resiliencia de dos niveles:
    1. Fallos de conexion/HTTP (timeout, DNS, codigos 429/500/502/503/504):
       gestionados por el adaptador urllib3 con reintento y backoff.
    2. Fallo "silencioso" (la API responde 200 OK pero el cuerpo no tiene
       la forma esperada): se fuerza un ValueError explicito.

    Si tras el intento no hay datos validos, recurre al ultimo lote Bronze
    real disponible para esa ciudad (FALLBACK). Si tampoco existe, propaga
    el error.

    Devuelve: tupla (cuerpo_json, origen) donde origen es "API" o "FALLBACK".
    """
    url = API_URL_BASE
    params = construir_parametros(lat, lon)
    headers = {"User-Agent": USER_AGENT_API}
    try:
        logger.info(f"Conexion a la API de Open-Meteo para {nombre_ciudad}: {url}")
        respuesta = sesion.get(url, params=params, headers=headers, timeout=API_TIMEOUT)
        respuesta.raise_for_status()
        cuerpo = respuesta.json()

        if "hourly" not in cuerpo or "time" not in cuerpo.get("hourly", {}):
            raise ValueError(
                "La API respondio 200 OK pero el cuerpo no tiene el formato "
                "esperado (falta la seccion 'hourly')."
            )
        if len(cuerpo["hourly"]["time"]) == 0:
            raise ValueError("La API respondio 200 OK pero la serie horaria esta vacia.")

        logger.info(
            f"Datos obtenidos desde la API para {nombre_ciudad}: "
            f"{len(cuerpo['hourly']['time'])} registros horarios"
        )
        return cuerpo, "API"

    except (requests.exceptions.RequestException, ValueError) as error:
        logger.warning(f"Fallo al obtener datos de Open-Meteo para {nombre_ciudad}: {error}")
        cuerpo_fallback = cargar_fallback_desde_bronze(nombre_ciudad)
        if cuerpo_fallback is not None:
            return cuerpo_fallback, "FALLBACK"
        logger.error(
            f"No fue posible obtener datos de la API para {nombre_ciudad} y no "
            f"existe ningun lote Bronze previo para usar como FALLBACK. "
            f"Se omite {nombre_ciudad} en esta ejecucion."
        )
        return None, "SIN_DATOS"


def obtener_datos_todas_las_ciudades(ubicaciones=None):
    """Ejecuta obtener_datos_ciudad() para cada ciudad del catalogo (o del
    subconjunto indicado en 'ubicaciones', usado por el CLI para
    seleccionar localizaciones concretas).

    Devuelve: diccionario {nombre_ciudad: (cuerpo_json_o_None, origen)}.
    """
    ubicaciones = ubicaciones if ubicaciones is not None else UBICACIONES
    sesion = crear_sesion_resiliente()
    resultados = {}
    for nombre_ciudad, coords in ubicaciones.items():
        resultados[nombre_ciudad] = obtener_datos_ciudad(
            sesion, nombre_ciudad, coords["lat"], coords["lon"]
        )
    return resultados

## Prueba rapida

Llamada real a `API_URL_BASE` para Madrid, para ver en vivo el comportamiento de reintento/fallback si la API falla o no hay red, y confirmar el origen final del dato (`API` o `FALLBACK`).

In [4]:
sesion_prueba = crear_sesion_resiliente()
coords_madrid = UBICACIONES["Madrid"]
cuerpo_prueba, origen_prueba = obtener_datos_ciudad(
    sesion_prueba, "Madrid", coords_madrid["lat"], coords_madrid["lon"]
)

print(f"Origen del dato: {origen_prueba}")
if cuerpo_prueba is not None:
    n_horas = len(cuerpo_prueba["hourly"]["time"])
    print(f"Registros horarios recibidos: {n_horas}")
    print(f"Primer registro: {cuerpo_prueba['hourly']['time'][0]}")
    print(f"Ultimo registro:  {cuerpo_prueba['hourly']['time'][-1]}")
    print("Ejemplo (primeras 3 horas):")
    for i in range(min(3, n_horas)):
        print(
            f"  {cuerpo_prueba['hourly']['time'][i]} | "
            f"temp={cuerpo_prueba['hourly']['temperature_2m'][i]} C | "
            f"hum={cuerpo_prueba['hourly']['relative_humidity_2m'][i]} % | "
            f"precip={cuerpo_prueba['hourly']['precipitation'][i]} mm | "
            f"viento={cuerpo_prueba['hourly']['wind_speed_10m'][i]} km/h | "
            f"presion={cuerpo_prueba['hourly']['surface_pressure'][i]} hPa"
        )
else:
    print(
        "Sin datos disponibles (ni API ni FALLBACK). Esto es esperable en "
        "una primera ejecucion sin red y sin ningun lote Bronze previo."
    )

Origen del dato: API
Registros horarios recibidos: 336
Primer registro: 2026-09-22T10:00
Ultimo registro:  2026-10-06T09:00
Ejemplo (primeras 3 horas):
  2026-09-22T10:00 | temp=16.9 C | hum=80.0 % | precip=0.0 mm | viento=8.0 km/h | presion=1015.0 hPa
  2026-09-22T11:00 | temp=17.1 C | hum=79.3 % | precip=0.05 mm | viento=8.0 km/h | presion=1015.1 hPa
  2026-09-22T12:00 | temp=17.7 C | hum=77.3 % | precip=0.09 mm | viento=8.1 km/h | presion=1015.2 hPa


---
✅ **`extract_openmeteo.ipynb` verificado/a.**